Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# OWL-v2 Fine-Tuning on Coffee & Cashew Datasets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/betum_tool/models/owlv2/notebooks/finetune.ipynb)

## 1. Setup

In [ ]:
# @title 1a. Install dependencies
!pip install -q transformers torch torchvision pycocotools Pillow matplotlib torchmetrics datasets ml_collections absl-py etils[epath]

In [ ]:
# @title 1b. Clone the repo (for common scripts)
import os

REPO_DIR = "google-research/betum_tool"

if not os.path.exists("google-research"):
    repo_url = "https://github.com/google-research/google-research.git"
    print("Cloning google-research monorepo (sparse checkout)...")
    # We do a sparse checkout of only the betum_tool directory to avoid downloading 2GB+ of monorepo code.
    !git clone --depth=1 --no-checkout {repo_url} google-research
    !cd google-research && git sparse-checkout set betum_tool && git checkout
    print("✓ Cloned google-research/betum_tool/")
else:
    print("Repo already cloned.")

# Verify common scripts exist
assert os.path.exists(f"{REPO_DIR}/common/yolo_to_coco.py"), "Missing yolo_to_coco.py"
assert os.path.exists(f"{REPO_DIR}/common/class_map.json"), "Missing class_map.json"
print("✓ Common scripts found and updated.")

In [ ]:
# @title 1c. Download dataset from Mendeley
import os
import subprocess
import requests

DATA_DIR = "data"
MENDELEY_URL = "https://data.mendeley.com/public-api/zip/r46c6bpfpf/download/1"
ZIP_NAME = "dataset.zip"

os.makedirs(DATA_DIR, exist_ok=True)

zip_path = os.path.join(DATA_DIR, ZIP_NAME)
if not os.path.exists(zip_path):
    print("Downloading dataset from Mendeley (this may take a few minutes)...")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        print("Attempting download with requests...")
        response = requests.get(MENDELEY_URL, headers=headers, stream=True)
        if response.status_code == 200:
            with open(zip_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"Downloaded to {zip_path}")
        else:
            print(f"HTTP Error {response.status_code} with requests.")
            raise Exception(f"HTTP Error {response.status_code}")
            
    except Exception as e:
        print(f"Requests failed: {e}")
        print("Trying fallback with wget...")
        try:
            subprocess.run(["wget", "-U", "Mozilla/5.0", "-O", zip_path, MENDELEY_URL], check=True)
            print(f"Downloaded with wget to {zip_path}")
        except Exception as wget_e:
            print(f"Wget also failed: {wget_e}")
            print("Please manually download the dataset from https://data.mendeley.com/datasets/r46c6bpfpf/1 and upload it to the 'data' folder.")
else:
    print(f"Dataset already downloaded at {zip_path}")

In [ ]:
# @title 1d. Unzip main archive
import zipfile
import os

DATA_DIR = "data"
ZIP_NAME = "dataset.zip"
zip_path = os.path.join(DATA_DIR, ZIP_NAME)

print("Unzipping main archive...")
try:
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print("Done!")
except Exception as e:
    print(f"Error unzipping {zip_path}: {e}")

# Debug: List contents to see what we got
print("Contents of DATA_DIR after unzip:")
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}[{os.path.basename(root)}/]")
    subindent = ' ' * 4 * (level + 1)
    for f in files[:5]: # Show first 5 files to avoid spam
        print(f"{subindent}{f}")
    if len(files) > 5:
        print(f"{subindent}... ({len(files) - 5} more files)")

In [ ]:
# @title 1e. Extract .rar files
import os
import subprocess

DATA_DIR = "data"

print("Searching for .rar files...")
rar_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.endswith(".rar"):
            rar_files.append(os.path.join(root, file))

print(f"Found rar files: {rar_files}")

for rar_path in rar_files:
    print(f"Extracting {rar_path}...")
    try:
        # Extract directly to DATA_DIR so that Cashew and Coffee folders 
        # end up in data/ instead of data/Coffee and Cashew Nut Dataset/
        subprocess.run(["unrar", "x", "-o+", rar_path, DATA_DIR], check=True)
        print(f"Extracted {rar_path} to {DATA_DIR}")
    except Exception as e:
        print(f"Error extracting {rar_path}: {e}")
        print("Please ensure 'unrar' is installed (e.g., '!apt-get install unrar' in a cell).")

In [ ]:
# @title 1f. Verify dataset structure
import os

DATA_DIR = "data"

print("Verifying structure...")
for expected in ["Cashew/Cashew-Uganda/images", "Coffee/Batch1/images"]:
    found = False
    for root, dirs, files in os.walk(DATA_DIR):
        if expected in root:
            found = True
            print(f"✓ Found {expected} at {root}")
            break
    if not found:
        print(f"Warning: Could not find expected directory structure containing '{expected}'")
        
print("✓ Verification check complete.")

In [ ]:
# @title 1g. Flatten Coffee + Convert YOLO → COCO JSON
import shutil
import subprocess
import sys

# --- Flatten Coffee batches ---
# flatten_coffee.py expects data/Coffee/ in the current directory
coffee_flat_dir = os.path.join(DATA_DIR, "Coffee_flattened")
if os.path.exists(coffee_flat_dir):
    print("Cleaning up existing Coffee_flattened directory to force re-flattening...")
    shutil.rmtree(coffee_flat_dir)

print("Flattening Coffee dataset...")
subprocess.run([sys.executable, os.path.join(REPO_DIR, "common/flatten_coffee.py")], check=True)

# --- Convert YOLO → COCO for Cashew ---
os.makedirs(os.path.join(DATA_DIR, "coco"), exist_ok=True)

cashew_coco = os.path.join(DATA_DIR, "coco", "cashew_val.json")
if not os.path.exists(cashew_coco):
    subprocess.run([
        sys.executable,
        os.path.join(REPO_DIR, "common/yolo_to_coco.py"),
        "--images", os.path.join(DATA_DIR, "Cashew/Cashew-Uganda/images"),
        "--labels", os.path.join(DATA_DIR, "Cashew/Cashew-Uganda/Labels"),
        "--class_map", os.path.join(REPO_DIR, "common/class_map.json"),
        "--dataset", "cashew",
        "--output", os.path.join(DATA_DIR, "coco/"),
        "--split_ratio", "0.8"
    ], check=True)
else:
    print("Cashew COCO JSON already exists.")

# --- Convert YOLO → COCO for Coffee ---
coffee_coco = os.path.join(DATA_DIR, "coco", "coffee_val.json")
if not os.path.exists(coffee_coco):
    subprocess.run([
        sys.executable,
        os.path.join(REPO_DIR, "common/yolo_to_coco.py"),
        "--images", os.path.join(DATA_DIR, "Coffee_flattened/images"),
        "--labels", os.path.join(DATA_DIR, "Coffee_flattened/labels"),
        "--class_map", os.path.join(REPO_DIR, "common/class_map.json"),
        "--dataset", "coffee",
        "--output", os.path.join(DATA_DIR, "coco/"),
        "--split_ratio", "0.8"
    ], check=True)
else:
    print("Coffee COCO JSON already exists.")

print("\n✓ COCO ground truth ready.")

### 1h. Visualize Ground Truth Bounding Boxes

Let's use the shared utility `common/visualize_coco.py` to draw ground truth bounding boxes on a few validation images to inspect the annotations before we proceed to fine-tuning.

In [ ]:
# @title 1h. Visualize Ground Truth Annotations
import sys
sys.path.append(REPO_DIR)

from common.visualize_coco import visualize

# Visualize 3 random ground truth validation images for Cashew
print("Visualizing cashew validation ground truth annotations:")
visualize(
    coco_json="data/coco/cashew_val.json",
    image_dir="data/Cashew/Cashew-Uganda/images",
    num_images=3,
    output_dir=None
)

## 2. Device Selection

In [ ]:
# @title 2. Device Selection
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    precision = torch.bfloat16
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    precision = torch.float32
else:
    device = torch.device("cpu")
    precision = torch.float32

print(f"Using device: {device} with precision: {precision}")

## 3. Convert COCO → HF Dataset

In [ ]:
# @title 3. Convert COCO → HF Dataset
import os
import shutil
import subprocess
import sys
import datasets

# The script expects train.json and valid.json in the root, and images in an 'images' subfolder.
# Let's create that structure using symlinks to avoid copying files.

COCO_HF_DIR = "data/coco_hf"
os.makedirs(COCO_HF_DIR, exist_ok=True)

# Symlink images
target_images = os.path.abspath("data/Cashew/Cashew-Uganda/images")
link_images = os.path.abspath(os.path.join(COCO_HF_DIR, "images"))
if not os.path.exists(link_images):
    os.symlink(target_images, link_images)

# Copy/Rename JSON files
shutil.copy("data/coco/cashew_train.json", os.path.join(COCO_HF_DIR, "train.json"))
shutil.copy("data/coco/cashew_val.json", os.path.join(COCO_HF_DIR, "valid.json"))

# Run the script using subprocess to ensure errors are correctly raised and visible in Jupyter/Colab
subprocess.run([
    sys.executable,
    os.path.join(REPO_DIR, "models/owlv2/scripts/convert_coco_json_to_hf_dataset.py"),
    f"--coco_root_dir={COCO_HF_DIR}",
    "--output_hf_path=data/hf"
], check=True)

# Load the dataset
dataset_dict = datasets.load_from_disk("data/hf")
cashew_train_dataset = dataset_dict["train"]
cashew_val_dataset = dataset_dict["valid"]

# Slice for smoke test if needed
if 'NUM_EXAMPLES' in globals() and NUM_EXAMPLES:
    cashew_train_dataset = cashew_train_dataset.select(range(min(NUM_EXAMPLES, len(cashew_train_dataset))))
    cashew_val_dataset = cashew_val_dataset.select(range(min(NUM_EXAMPLES, len(cashew_val_dataset))))

print(f"Cashew train: {len(cashew_train_dataset)} samples")
print(f"Cashew val: {len(cashew_val_dataset)} samples")

## 4. Load Model & Processor

In [ ]:
# @title 4. Load Model & Processor
from transformers import Owlv2ForObjectDetection, Owlv2Processor

MODEL_ID = "google/owlv2-base-patch16-ensemble"
print(f"Loading model {MODEL_ID}...")
processor = Owlv2Processor.from_pretrained(MODEL_ID)
model = Owlv2ForObjectDetection.from_pretrained(MODEL_ID)
model.to(device)
print("✓ Model loaded.")

## 5. Define Class Mapping

In [ ]:
# @title 5. Define Class Mapping
# Get class names from dataset features
class_names = cashew_train_dataset.features["objects"]["category"].feature.names
print(f"Class names: {class_names}")

# Build text queries for OWL-v2
# We can reuse the prompts from the plan or inference notebook
text_queries = [f"a {name}" for name in class_names]
print(f"Text queries: {text_queries}")

## 6. Apply Transforms

In [ ]:
# @title 6. Apply Transforms
import sys
sys.path.append("Betum_Tool")

from models.owlv2.owl import Owlv2Engine

# Create label mappings
dataset_id2label = {i: name for i, name in enumerate(class_names)}
model_label2id = {name: i for i, name in enumerate(class_names)}

engine = Owlv2Engine()
transform_fn = engine.get_transform_fn(processor, text_queries, dataset_id2label, model_label2id)

# Apply to datasets
cashew_train_dataset.set_transform(transform_fn)
cashew_val_dataset.set_transform(transform_fn)

print("✓ Transforms applied.")

### 6b. Collation Function

In [ ]:
# @title 6b. Collation Function
from models.owlv2.config import get_base_config, Precision
import torch

cfg = get_base_config()
cfg.training.precision = Precision.BF16 if torch.cuda.is_available() else Precision.FP32

collate_fn = engine.get_collate_fn(cfg)
print("✓ Collation function ready.")

## 7. Set Up Loss & Matcher

In [ ]:
# @title 7. Set Up Loss & Matcher
from models.owlv2.losses import SetCriterion
from models.owlv2.matcher import HungarianMatcher

matcher = HungarianMatcher(cost_class=1.0, cost_bbox=1.0, cost_giou=1.0)
weight_dict = {"loss_sigmoid_focal": 1.0, "loss_bbox": 1.0, "loss_giou": 1.0}

criterion = SetCriterion(
    num_classes=len(class_names),
    matcher=matcher,
    weight_dict=weight_dict,
    eos_coef=0.1,
    losses=["labels", "boxes"],
    focal_alpha=0.25,
    focal_gamma=2.0,
)

print("✓ Loss and Matcher ready.")

## 8. Configure Training

In [ ]:
# @title 8. Configure Training
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,           # smoke-test; increase for real runs
    gradient_accumulation_steps=2, # maintains effective batch size of 4
    gradient_checkpointing=True,  # saves memory
    learning_rate=1e-5,
    bf16=torch.cuda.is_available(),
    remove_unused_columns=False,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=10,
)

print("✓ Training arguments configured.")

### 8b. Metrics

In [ ]:
# @title 8b. Metrics (Unified Evaluation Post-Training)
print("✓ We will run unified post-inference evaluation at the end of training.")

## 9. Train

In [ ]:
# @title 9. Train
import inspect
import transformers
from models.owlv2.trainer import CustomTrainer

# Dynamically determine whether to use "processing_class" or legacy "tokenizer" parameter
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": cashew_train_dataset,
    "eval_dataset": cashew_val_dataset,
    "data_collator": collate_fn,
    "criterion": criterion,
    "weight_dict": weight_dict,
}

trainer_sig = inspect.signature(transformers.Trainer.__init__)
if "processing_class" in trainer_sig.parameters:
    trainer_kwargs["processing_class"] = processor
else:
    trainer_kwargs["tokenizer"] = processor

trainer = CustomTrainer(**trainer_kwargs)

print("Starting training...")
trainer.train()
print("✓ Training complete.")

## 10. Evaluate

In [ ]:
# @title 10. Save Model Checkpoint
trainer.save_model("./results/finetuned_model")
print("✓ Saved fine-tuned model checkpoint.")

## 11. Run Inference on Validation Set

Now we run inference on all images in the validation split using our fine-tuned OWL-v2 model, format all predictions as standard COCO bounding boxes, and save them for evaluation and visualization.

In [ ]:
# @title 11. Full Validation Inference
from PIL import Image
from pathlib import Path
import json
from tqdm import tqdm

# Set model to evaluation mode
trainer.model.eval()

# Define the inference helper
def run_owlv2_inference(
    model, processor, image, text_queries, device, score_threshold=0.01
):
    inputs = processor(
        text=text_queries, images=image, return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[0]  # [N, num_classes]
    pred_boxes = outputs.pred_boxes[0]  # [N, 4]

    probs = logits.sigmoid()
    scores, labels = probs.max(dim=-1)

    mask = scores > score_threshold
    filtered_boxes = pred_boxes[mask].cpu().tolist()
    filtered_scores = scores[mask].cpu().tolist()
    filtered_labels = labels[mask].cpu().tolist()

    return {
        "boxes": filtered_boxes,
        "scores": filtered_scores,
        "labels": filtered_labels,
    }

# Define box conversion helper
def owlv2_box_to_coco(cx, cy, w, h, img_width, img_height):
    max_side = max(img_width, img_height)
    abs_cx = cx * max_side
    abs_cy = cy * max_side
    abs_w = w * max_side
    abs_h = h * max_side

    x_min = max(0.0, abs_cx - abs_w / 2)
    y_min = max(0.0, abs_cy - abs_h / 2)
    abs_w = min(abs_w, float(img_width) - x_min)
    abs_h = min(abs_h, float(img_height) - y_min)

    return [round(x_min, 2), round(y_min, 2), round(abs_w, 2), round(abs_h, 2)]

# Load ground truth COCO for visualising
with open("data/coco/cashew_val.json", "r") as f:
    coco_gt_data = json.load(f)

images_info = coco_gt_data["images"]
categories = {cat["id"]: cat["name"] for cat in coco_gt_data["categories"]}

# Setup text queries
queries = [
    "a cashew tree",
    "a cashew flower",
    "a premature cashew nut",
    "an unripe cashew nut",
    "a ripe cashew nut",
    "a spoilt cashew nut",
]

fine_tuned_predictions = []
print(f"Running inference on {len(images_info)} validation images using fine-tuned model...")

for img_info in tqdm(images_info):
    img_path = Path("data/Cashew/Cashew-Uganda/images") / img_info["file_name"]
    if not img_path.exists():
        img_path = Path("data/Cashew/Cashew-Uganda/images") / img_info["file_name"].strip()
    
    image = Image.open(img_path).convert("RGB")
    
    result = run_owlv2_inference(
        trainer.model, processor, image, queries, device,
        score_threshold=0.01  # Collect all candidates
    )
    
    for box, score, label in zip(result["boxes"], result["scores"], result["labels"]):
        coco_box = owlv2_box_to_coco(
            box[0], box[1], box[2], box[3], img_info["width"], img_info["height"]
        )
        fine_tuned_predictions.append({
            "image_id": img_info["id"],
            "category_id": label,
            "bbox": coco_box,
            "score": score
        })

# Save predictions to disk
output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)
predictions_path = output_dir / "cashew_finetuned_predictions.json"
with open(predictions_path, "w") as f:
    json.dump(fine_tuned_predictions, f, indent=2)

print(f"\n✓ Saved predictions to {predictions_path}")

## 12. Unified Evaluation (COCOeval)

Now we run standard COCO evaluation on all predictions to measure average precision (AP@50) across overall and per-category metrics.

In [ ]:
# @title 12. Unified Evaluation
import sys

# Append repo path to sys path to load common
if 'REPO_DIR' in globals() and REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)
from common.evaluate import evaluate_coco

# Evaluate predictions with default threshold
print("=== Running Unified Evaluation ===")
results = evaluate_coco(
    gt_json="data/coco/cashew_val.json",
    predictions=predictions_path,
    score_threshold=0.15
)

## 13. Visualise Predictions vs Ground Truth

To qualitatively assess how well your fine-tuned OWL-v2 model performs, let's visualize predictions vs ground truth side-by-side on a few sample images from the validation set using our saved predictions.

In [ ]:
# @title 13. Side-by-Side Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random

VIS_THRESHOLD = 0.3
NUM_VIS = 3

# Group GT annotations
gt_by_image = {}
for ann in coco_gt_data["annotations"]:
    img_id = ann["image_id"]
    if img_id not in gt_by_image:
        gt_by_image[img_id] = []
    gt_by_image[img_id].append(ann)

# Group predictions
pred_by_image = {}
for pred in fine_tuned_predictions:
    if pred["score"] < VIS_THRESHOLD:
        continue
    img_id = pred["image_id"]
    if img_id not in pred_by_image:
        pred_by_image[img_id] = []
    pred_by_image[img_id].append(pred)

# Set color map
cmap = plt.cm.get_cmap("tab10")
cat_colors = {cat_id: cmap(i % 10) for i, cat_id in enumerate(categories.keys())}

def draw_boxes(ax, boxes, is_gt=True):
    for item in boxes:
        bbox = item["bbox"]
        cat_id = item["category_id"]
        color = cat_colors.get(cat_id, "red")
        linestyle = "-" if is_gt else "--"
        linewidth = 2 if is_gt else 1.5
        
        rect = patches.Rectangle(
            (bbox[0], bbox[1]), bbox[2], bbox[3],
            linewidth=linewidth, edgecolor=color,
            facecolor="none", linestyle=linestyle
        )
        ax.add_patch(rect)
        
        label = categories.get(cat_id, f"cls{cat_id}")
        if not is_gt and "score" in item:
            label = f"{label} {item['score']:.2f}"
        
        ax.text(
            bbox[0], bbox[1] - 4, label,
            fontsize=7, color=color,
            bbox=dict(boxstyle="round,pad=0.15", facecolor="black", alpha=0.6)
        )

# Sample random validation images
sample_images = random.sample(images_info, min(NUM_VIS, len(images_info)))

fig, axes = plt.subplots(len(sample_images), 2, figsize=(16, 5 * len(sample_images)))
if len(sample_images) == 1:
    axes = axes.reshape(1, -1)

for row, img_info in enumerate(sample_images):
    img_id = img_info["id"]
    img_path = Path("data/Cashew/Cashew-Uganda/images") / img_info["file_name"]
    if not img_path.exists():
        img_path = Path("data/Cashew/Cashew-Uganda/images") / img_info["file_name"].strip()
        
    image = Image.open(img_path).convert("RGB")
    
    # Ground truth
    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"Ground Truth — {img_info['file_name']}", fontsize=10)
    axes[row, 0].axis("off")
    draw_boxes(axes[row, 0], gt_by_image.get(img_id, []), is_gt=True)
    
    # Predictions (pre-computed!)
    axes[row, 1].imshow(image)
    axes[row, 1].set_title(f"Fine-tuned OWL-v2 Predictions (vis_threshold={VIS_THRESHOLD})", fontsize=10)
    axes[row, 1].axis("off")
    draw_boxes(axes[row, 1], pred_by_image.get(img_id, []), is_gt=False)

plt.tight_layout()
plt.show()